# 1. Train an isotopomer model

This notebook trains a neural network to predict an isotopomer distribution from HSQC + GC-MS data, for a single **HSQC vector**.

The pipeline is:

1. **Simulate** many synthetic isotopomer distributions and their HSQC/GC-MS signatures.
2. **Collate** them into feature vectors `X` and label vectors `Y`.
3. **Train** a small fixed-architecture network (fast — no hyperparameter search).
4. **Save** the model so `02_predict.ipynb` can use it on real data.

> For a one-command version of exactly this, run `mlp-train --hsqc-vector 0 1 1` from a terminal.

In [ ]:
from metabolabpytools import isotopomerAnalysis

analysis = isotopomerAnalysis.IsotopomerAnalysisNN()

# The HSQC vector: 1 = carbon observed in HSQC, 0 = not observed.
hsqc_vector = [0, 1, 1]
n_carbons = len(hsqc_vector)

# More carbons -> 2**n_carbons output classes -> more training samples needed.
n_distributions = 10000 if n_carbons <= 3 else (20000 if n_carbons == 4 else 40000)
n_distributions

## Step 1 — Simulate training data

Generate random isotopomer distributions and simulate the HSQC multiplet and GC-MS percentages each would produce. A copy is saved to `sim_data/` for reference.

In [ ]:
distributions = analysis.generate_isotopomer_distributions(n_distributions=n_distributions, n_carbons=n_carbons)
isotopomer_data, hsqc_data, gcms_data = analysis.simulate_hsqc_gcms(distributions, hsqc_vector)
analysis.save_simulation_data(isotopomer_data, hsqc_data, gcms_data, hsqc_vector)

## Step 2 — Collate features and labels

`X` = HSQC multiplet percentages (ordered against every possible multiplet for this vector) concatenated with GC-MS percentages. `Y` = the true isotopomer percentages.

In [ ]:
all_possible_hsqc_multiplets = analysis.generate_possible_hsqc_multiplets(hsqc_vector)
Y = analysis.collate_y_labels(isotopomer_data, n_carbons)
X = analysis.collate_x_labels_without_noise(hsqc_data, gcms_data, all_possible_hsqc_multiplets)

print('X shape:', X.shape, ' Y shape:', Y.shape)
print('example X:', X[7])
print('example Y:', Y[7])

## Step 3 — Train and save

A fixed small network (two hidden layers) with early stopping. Passing `hsqc_vector` saves the model to `saved_models/` and a summary to `model_summaries/` automatically.

In [ ]:
model, history = analysis.train_neural_network(X, Y, hsqc_vector=hsqc_vector, epochs=500, batch_size=64)